# Heat Risk Index — Sensitivity Analysis & VCP Comparison

Four analyses:

1. **Weighting sensitivity** — compare facility rankings under three component-weighting schemes (current/historic period only)
2. **Own-tract VCP validation** — correlate our risk score against VCP's `ExHeatHealth_Idx` for each prison's host census tract
3. **Surrounding community comparison** — adjacent non-institutional tract VCP heat values vs. our index, as a methodological argument for a prison-specific index
4. **Hazard design-parameter sensitivity** — rank stability across sweeps of β (AQI multiplier) and W_REL (daytime relative/absolute blend weight)

Output: `data/cdcr/CDCR_heat_risk_sensitivity.csv`

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from scipy import stats

# Load risk index — mid-century only
risk = pd.read_csv('data/cdcr/CDCR_heat_risk_index_additive_25_25_50.csv')
mc = risk[risk['time_period'] == 'current'].copy().reset_index(drop=True)
print(f'Facilities: {len(mc)}')
print(mc[['cdcr_code', 'hazard_score', 'exposure_score', 'vulnerability_score', 'risk_score']].to_string(index=False))

# Load facilities for tract_geoid join
fac = pd.read_csv('data/cdcr/cdcr_facilities.csv')[['cdcr_code', 'tract_geoid']]
mc = mc.merge(fac, on='cdcr_code', how='left')
mc['tract_str'] = mc['tract_geoid'].astype(str).str.split('.').str[0].str.zfill(11)
print(f'\ntract_geoid nulls: {mc["tract_geoid"].isnull().sum()}')

Facilities: 31
cdcr_code  hazard_score  exposure_score  vulnerability_score  risk_score
      ASP          0.31            0.47                 0.16       43.23
      CAL          0.39            0.22                 0.18       38.26
      CCI          0.22            0.50                 0.26       48.46
     CCWF          0.33            0.43                 0.42       62.80
      CEN          0.40            0.07                 0.20       34.01
     CHCF          0.33            0.09                 0.60       63.19
      CIM          0.37            0.66                 0.44       75.30
      CIW          0.36            0.56                 0.41       68.22
      CMC          0.27            0.28                 0.41       54.21
      CMF          0.32            0.49                 0.55       74.57
      COR          0.36            0.66                 0.38       69.77
      CRC          0.35            0.62                 0.25       57.49
      CTF          0.26            0

## 1. Weighting Sensitivity

Three schemes tested against additive vulnerability-upweighted (current index):

| Scheme | Formula | Logic |
|---|---|---|
| A — Additive 25/25/50 (current) | 0.25×H + 0.25×E + 0.50×V, normalized 0–100 | Ovienmhada vulnerability upweighting; additive — high vulnerability alone can drive rank |
| B — Equal multiplicative | H × E × V, normalized 0–100 | All components equal; risk requires elevation in all three |
| C — Multiplicative V² | H × E × V², normalized 0–100 | Preserves multiplicative structure; vulnerability amplified but still requires hazard and exposure |

Scores for all three are normalized to 0–100 within mid-century facilities only.

In [2]:
H = mc['hazard_score']
E = mc['exposure_score']
V = mc['vulnerability_score']

def norm100(s):
    mn, mx = s.min(), s.max()
    return (s - mn) / (mx - mn) * 100

# Scheme A: additive 25/25/50 (current index default)
mc['score_A'] = norm100(0.25 * H + 0.25 * E + 0.50 * V)

# Scheme B: equal multiplicative
mc['score_B'] = norm100(H * E * V)

# Scheme C: multiplicative V²
mc['score_C'] = norm100(H * E * V**2)

# Ranks (1 = highest risk)
for col, rank_col in [('score_A', 'rank_A'), ('score_B', 'rank_B'), ('score_C', 'rank_C')]:
    mc[rank_col] = mc[col].rank(ascending=False).astype(int)

# Spearman correlations between schemes
pairs = [('A', 'B'), ('A', 'C'), ('B', 'C')]
print('Spearman rank correlations (mid-century):')
for s1, s2 in pairs:
    r, p = stats.spearmanr(mc[f'rank_{s1}'], mc[f'rank_{s2}'])
    print(f'  {s1} vs {s2}: r={r:.3f}, p={p:.4f}')

print()

# Full rank comparison table
rank_cols = ['cdcr_code', 'rank_A', 'rank_B', 'rank_C']
rank_df = mc[rank_cols].copy()
rank_df['max_swing'] = rank_df[['rank_A','rank_B','rank_C']].max(axis=1) - rank_df[['rank_A','rank_B','rank_C']].min(axis=1)
rank_df = rank_df.sort_values('rank_A')
print('Rankings by scheme (sorted by Scheme A):')
print(rank_df.to_string(index=False))

Spearman rank correlations (mid-century):
  A vs B: r=0.893, p=0.0000
  A vs C: r=0.961, p=0.0000
  B vs C: r=0.919, p=0.0000

Rankings by scheme (sorted by Scheme A):
cdcr_code  rank_A  rank_B  rank_C  max_swing
      CMF       1       4       1          3
      CIM       1       1       2          1
      COR       3       2       4          2
      SAC       4       6       5          2
     SATF       4       3       3          1
      CIW       6       5       6          1
      RJD       7      10       7          3
     CHCF       8      27      17         19
     CCWF       9       8       8          1
      SOL      10       7      11          4
      LAC      11      13      10          3
      VSP      12      11       9          3
      CRC      13       9      14          5
     MCSP      14      16      12          4
     PBSP      15      17      18          3
     KVSP      16      12      13          4
      CMC      17      22      16          6
     NKSP      17     

In [3]:
# Facilities with largest rank swings across all three schemes
print('Facilities with max rank swing ≥ 5 positions:')
large_swings = rank_df[rank_df['max_swing'] >= 5].sort_values('max_swing', ascending=False)
if len(large_swings) == 0:
    print('  None — all facilities stable within 4 rank positions across schemes')
else:
    print(large_swings.to_string(index=False))

print()
print('Top 10 by Scheme A (current equal-weight):')
print(mc[['cdcr_code','name','score_A','score_B','score_C','rank_A','rank_B','rank_C']]
      .sort_values('rank_A').head(10).round(1).to_string(index=False))

Facilities with max rank swing ≥ 5 positions:
cdcr_code  rank_A  rank_B  rank_C  max_swing
     CHCF       8      27      17         19
     NKSP      17      14      22          8
      WSP      20      15      23          8
      CMC      17      22      16          6
      CRC      13       9      14          5

Top 10 by Scheme A (current equal-weight):
cdcr_code                                  name  score_A  score_B  score_C  rank_A  rank_B  rank_C
      CMF           California Medical Facility    100.0     80.3    100.0       1       4       1
      CIM        California Institution For Men    100.0    100.0     99.7       1       1       2
      COR     California State Prison, Corcoran     87.5     84.0     72.3       3       2       4
      SAC   California State Prison, Sacramento     85.6     61.9     71.6       4       6       5
     SATF Ca Substance Abuse Treatment Facility     85.6     80.4     72.9       4       3       3
      CIW      California Institution For Wome

## 2. Own-Tract VCP Comparison

Correlates our mid-century risk score against VCP's `ExHeatHealth_Idx` for each prison's host census tract.

`ExHeatHealth_Idx` is VCP's combined extreme heat health index — a composite of heat hazard (days over 100°F, hot nights) and social vulnerability (chronic disease, income, age, disability). It is available for all census tracts including prison tracts.

**Expected finding:** Low correlation. VCP's social vulnerability component uses community-level demographics (income, renters, limited English households) that do not describe the incarcerated population. This divergence is the methodological argument for a prison-specific index rather than a limitation of ours.

In [4]:
# Load VCP
vcp = gpd.read_file('data_sources/hazards/VCP_Tracts.geojson')
vcp['GEOID_str'] = vcp['GEOID'].astype(str)

vcp_cols = ['GEOID_str', 'Pct_GroupQuarters', 'ExHeatHealth_Idx',
            'Heat_sc_hazard_pre', 'Heat_sc_hazard_fut', 'Heat_sc_social',
            'INDEX_PCTL', 'geometry']
vcp_slim = vcp[vcp_cols].copy()

# Join prison tracts
mc_vcp = mc.merge(
    vcp_slim.drop(columns='geometry'),
    left_on='tract_str', right_on='GEOID_str', how='left'
)

n_matched = mc_vcp['ExHeatHealth_Idx'].notna().sum()
print(f'Facilities matched to VCP tract: {n_matched} of {len(mc_vcp)}')
print(f'Pct_GroupQuarters for prison tracts:')
print(mc_vcp[['cdcr_code', 'tract_str', 'Pct_GroupQuarters', 'ExHeatHealth_Idx', 'INDEX_PCTL']]
      .sort_values('Pct_GroupQuarters', ascending=False).to_string(index=False))

print()
# Pearson and Spearman vs our risk score
pair = mc_vcp[['risk_score', 'ExHeatHealth_Idx']].dropna()
r_p, p_p = stats.pearsonr(pair['risk_score'], pair['ExHeatHealth_Idx'])
r_s, p_s = stats.spearmanr(pair['risk_score'], pair['ExHeatHealth_Idx'])
print(f'Our risk_score vs VCP ExHeatHealth_Idx (n={len(pair)}):')
print(f'  Pearson  r={r_p:.3f}, p={p_p:.4f}')
print(f'  Spearman r={r_s:.3f}, p={p_s:.4f}')

# Also compare just hazard components where available
pair_h = mc_vcp[['hazard_score', 'Heat_sc_hazard_fut']].dropna()
r_h, p_h = stats.spearmanr(pair_h['hazard_score'], pair_h['Heat_sc_hazard_fut'])
print(f'\nOur hazard_score vs VCP Heat_sc_hazard_fut (n={len(pair_h)}):')
print(f'  Spearman r={r_h:.3f}, p={p_h:.4f}')

Facilities matched to VCP tract: 31 of 31
Pct_GroupQuarters for prison tracts:
cdcr_code   tract_str  Pct_GroupQuarters  ExHeatHealth_Idx  INDEX_PCTL
      ASP 06031981800         100.000000         66.713737         NaN
      CTF 06053010900         100.000000         54.174316         NaN
     SVSP 06053010900         100.000000         54.174316         NaN
       SQ 06041122000         100.000000         60.360869         NaN
      SOL 06095253000         100.000000         71.025339         NaN
      SCC 06109985202         100.000000         34.594807         NaN
     SATF 06031980100         100.000000         79.570095         NaN
      RJD 06073010016         100.000000         51.240292         NaN
     NKSP 06029004601         100.000000         46.567820         NaN
      LAC 06037901003         100.000000         50.090217         NaN
     KVSP 06029004603         100.000000         55.021574         NaN
      ISP 06065981000         100.000000         53.669099         Na

## 3. Surrounding Community Comparison

For each prison, find adjacent VCP census tracts (sharing a border), exclude tracts with `Pct_GroupQuarters > 25%` (other institutional tracts), and average their `ExHeatHealth_Idx`.

This answers: *How does VCP assess heat risk for the communities surrounding each prison?* The comparison between that community-facing index and our prison-specific index illustrates why a dedicated framework is needed — VCP's community variables (income, renters, chronic disease prevalence in free populations) do not translate to the incarcerated context.

In [5]:
# Build spatial adjacency: for each prison tract, find touching VCP tracts
# vcp is already a GeoDataFrame with geometry

# Ensure consistent CRS
vcp_geo = vcp[['GEOID_str', 'Pct_GroupQuarters', 'ExHeatHealth_Idx', 'geometry']].copy()
vcp_geo = vcp_geo.set_index('GEOID_str')

community_rows = []

for _, row in mc_vcp.iterrows():
    tract = row['tract_str']
    if tract not in vcp_geo.index:
        community_rows.append({'cdcr_code': row['cdcr_code'], 'n_neighbors': 0,
                                'community_ExHeatHealth_Idx': np.nan})
        continue

    prison_geom = vcp_geo.loc[tract, 'geometry']

    # Find tracts that touch this one (shared border, not just point)
    neighbors = vcp_geo[
        (vcp_geo.index != tract) &
        (vcp_geo['geometry'].touches(prison_geom) | vcp_geo['geometry'].intersects(prison_geom)) &
        (vcp_geo.index != tract)
    ].copy()

    # Exclude other institutional tracts
    community = neighbors[neighbors['Pct_GroupQuarters'] <= 25]

    n = len(community)
    avg_idx = community['ExHeatHealth_Idx'].mean() if n > 0 else np.nan

    community_rows.append({
        'cdcr_code': row['cdcr_code'],
        'n_neighbors': n,
        'community_ExHeatHealth_Idx': avg_idx
    })

community_df = pd.DataFrame(community_rows)
mc_vcp = mc_vcp.merge(community_df, on='cdcr_code', how='left')

print('Community adjacent tract count and avg ExHeatHealth_Idx:')
print(mc_vcp[['cdcr_code', 'n_neighbors', 'community_ExHeatHealth_Idx', 'ExHeatHealth_Idx']]
      .sort_values('community_ExHeatHealth_Idx', ascending=False).to_string(index=False))

Community adjacent tract count and avg ExHeatHealth_Idx:
cdcr_code  n_neighbors  community_ExHeatHealth_Idx  ExHeatHealth_Idx
      CAL            4                   86.878481         92.161293
     SATF            4                   86.104966         79.570095
      COR            4                   86.104966         79.570095
      ISP            1                   86.034361         53.669099
     NKSP            2                   83.301169         46.567820
      SCC            1                   82.097748         34.594807
     PBSP            3                   80.410554         59.745823
     KVSP            1                   78.481211         55.021574
     PVSP            9                   77.445848         83.128579
      CEN           10                   76.296697         69.150388
     HDSP            4                   74.970189         33.471405
      VSP            7                   72.664941         82.797521
     CCWF            7                   72.66

In [6]:
# Summary comparison: our risk rank vs community VCP rank
mc_vcp['community_rank'] = mc_vcp['community_ExHeatHealth_Idx'].rank(ascending=False)

print('Prison risk rank (our index) vs surrounding community heat rank (VCP):')
compare = mc_vcp[['cdcr_code', 'name', 'risk_score', 'rank_A',
                   'community_ExHeatHealth_Idx', 'community_rank', 'n_neighbors']].copy()
compare['rank_diff'] = (compare['rank_A'] - compare['community_rank']).round(0).astype('Int64')
compare = compare.sort_values('rank_A')
print(compare.to_string(index=False))

print()
# Correlation between our risk rank and community VCP rank
pair_c = compare[['risk_score', 'community_ExHeatHealth_Idx']].dropna()
r_c, p_c = stats.spearmanr(pair_c['risk_score'], pair_c['community_ExHeatHealth_Idx'])
print(f'Spearman r (our risk_score vs community ExHeatHealth_Idx): r={r_c:.3f}, p={p_c:.4f} (n={len(pair_c)})')
print()
print('Positive rank_diff = our index ranks facility HIGHER risk than VCP ranks surrounding community')
print('Negative rank_diff = VCP ranks surrounding community higher than our index ranks the prison')

Prison risk rank (our index) vs surrounding community heat rank (VCP):
cdcr_code                                       name  risk_score  rank_A  community_ExHeatHealth_Idx  community_rank  n_neighbors  rank_diff
      CMF                California Medical Facility       74.57       1                   56.358751            21.5            4        -20
      CIM             California Institution For Men       75.30       1                   26.231158            29.0            7        -28
      COR          California State Prison, Corcoran       69.77       3                   86.104966             2.5            4          0
      SAC        California State Prison, Sacramento       69.26       4                   54.579901            24.5            4        -20
     SATF      Ca Substance Abuse Treatment Facility       69.44       4                   86.104966             2.5            4          2
      CIW           California Institution For Women       68.22       6           

## 4. Output CSV

Saves per-facility scores and ranks under all three weighting schemes, plus VCP comparison values, to `data/cdcr/CDCR_heat_risk_sensitivity.csv`.

In [7]:
# pct_hu_mechanical (CDCR Jan-2026 report) rides along on the index CSV via mc.
out_cols = [
    'cdcr_code', 'name', 'average_2025_population', 'pct_hu_mechanical',
    'hazard_score', 'exposure_score', 'vulnerability_score',
    # Weighting sensitivity
    'score_A', 'score_B', 'score_C',
    'rank_A', 'rank_B', 'rank_C',
    # VCP own-tract
    'tract_str', 'Pct_GroupQuarters', 'ExHeatHealth_Idx',
    'Heat_sc_hazard_fut', 'INDEX_PCTL',
    # VCP surrounding community
    'n_neighbors', 'community_ExHeatHealth_Idx',
]

output = mc_vcp[out_cols].copy()
output = output.rename(columns={
    'tract_str': 'tract_geoid',
    'score_A': 'risk_score_additive_25_25_50',
    'score_B': 'risk_score_equal_mult',
    'score_C': 'risk_score_mult_vsq',
    'rank_A':  'rank_additive_25_25_50',
    'rank_B':  'rank_equal_mult',
    'rank_C':  'rank_mult_vsq',
    'ExHeatHealth_Idx': 'vcp_own_tract_ExHeatHealth_Idx',
    'Heat_sc_hazard_fut': 'vcp_own_tract_heat_hazard_fut',
    'INDEX_PCTL': 'vcp_own_tract_INDEX_PCTL',
    'community_ExHeatHealth_Idx': 'vcp_community_ExHeatHealth_Idx',
})

score_cols = ['risk_score_additive_25_25_50', 'risk_score_equal_mult', 'risk_score_mult_vsq',
              'vcp_own_tract_ExHeatHealth_Idx', 'vcp_own_tract_heat_hazard_fut',
              'vcp_own_tract_INDEX_PCTL', 'vcp_community_ExHeatHealth_Idx']
output[score_cols] = output[score_cols].round(2)

# Self-identifying version tag, consistent with the index CSV.
output['index_version'] = 'v0.3'

output.to_csv('data/cdcr/CDCR_heat_risk_sensitivity.csv', index=False)
print(f'Saved {len(output)} rows to data/cdcr/CDCR_heat_risk_sensitivity.csv')
print(f'Columns: {list(output.columns)}')

Saved 31 rows to data/cdcr/CDCR_heat_risk_sensitivity.csv
Columns: ['cdcr_code', 'name', 'average_2025_population', 'pct_hu_mechanical', 'hazard_score', 'exposure_score', 'vulnerability_score', 'risk_score_additive_25_25_50', 'risk_score_equal_mult', 'risk_score_mult_vsq', 'rank_additive_25_25_50', 'rank_equal_mult', 'rank_mult_vsq', 'tract_geoid', 'Pct_GroupQuarters', 'vcp_own_tract_ExHeatHealth_Idx', 'vcp_own_tract_heat_hazard_fut', 'vcp_own_tract_INDEX_PCTL', 'n_neighbors', 'vcp_community_ExHeatHealth_Idx', 'index_version']


## 5. Hazard design-parameter sensitivity (β, W_REL)

The hazard has two design parameters: β (the AQI multiplier, default 0.30) and W_REL (the daytime relative/absolute blend weight, default 0.50). This recomputes the mid-century ranking across a sweep of each, holding exposure and vulnerability fixed, and reports the Spearman correlation and largest single-facility rank move against the defaults.

In [8]:
# Recompute the mid-century ranking across a sweep of each hazard design parameter,
# holding exposure and vulnerability fixed, and report rank stability vs the defaults
# (beta = 0.30, W_REL = 0.50).
from scipy import stats as _st

haz = pd.read_csv('data/hazards/heat_air_hazard.csv')
idxdf = pd.read_csv('data/cdcr/CDCR_heat_risk_index_additive_25_25_50.csv')
midx = idxdf[idxdf.time_period == 'midcentury'].set_index('cdcr_code')
Hh = haz[haz.cdcr_code.isin(midx.index)].set_index('cdcr_code')
Ecol, Vcol = midx['exposure_score'], midx['vulnerability_score']

def _hazard_mid(beta, w_rel):
    rel_max = Hh[['loca2_days_over_avg_plus10_historic', 'loca2_days_over_avg_plus10_midcentury']].to_numpy().max()
    abs_max = Hh[['loca2_days_over_90_historic', 'loca2_days_over_90_midcentury']].to_numpy().max()
    ngt_max = Hh[['loca2_nights_over_p95_historic', 'loca2_nights_over_p95_midcentury']].to_numpy().max()
    Hs = {}
    for key, suf in [('c', 'historic'), ('m', 'midcentury')]:
        day = w_rel * (Hh[f'loca2_days_over_avg_plus10_{suf}'] / rel_max) + (1 - w_rel) * (Hh[f'loca2_days_over_90_{suf}'] / abs_max)
        night = Hh[f'loca2_nights_over_p95_{suf}'] / ngt_max
        Hs[key] = ((day + night) / 2) * (1 + beta * Hh['AQI_norm'].fillna(0) / 100)
    return Hs['m'] / pd.concat(Hs.values()).max()

def _risk_rank(beta, w_rel):
    hm = _hazard_mid(beta, w_rel).reindex(Ecol.index)
    return (0.25 * hm + 0.25 * Ecol + 0.50 * Vcol).rank(ascending=False)

base = _risk_rank(0.30, 0.50)
print('AQI beta sensitivity (W_REL=0.50) — Spearman vs beta=0.30, largest single rank move:')
for b in [0.0, 0.15, 0.30, 0.50]:
    rk = _risk_rank(b, 0.50); d = (rk - base).abs()
    print(f'  beta={b:.2f}: rho={_st.spearmanr(base, rk)[0]:.3f}  max move={int(d.max())} ({d.idxmax()})')

print('\nDaytime blend W_REL sensitivity (beta=0.30) — Spearman vs W_REL=0.50, largest single rank move:')
for w in [0.0, 0.25, 0.50, 0.75, 1.0]:
    rk = _risk_rank(0.30, w); d = (rk - base).abs()
    print(f'  W_REL={w:.2f}: rho={_st.spearmanr(base, rk)[0]:.3f}  max move={int(d.max())} ({d.idxmax()})')

AQI beta sensitivity (W_REL=0.50) — Spearman vs beta=0.30, largest single rank move:
  beta=0.00: rho=0.994  max move=3 (WSP)
  beta=0.15: rho=0.997  max move=2 (HDSP)
  beta=0.30: rho=1.000  max move=0 (ASP)
  beta=0.50: rho=0.997  max move=2 (LAC)

Daytime blend W_REL sensitivity (beta=0.30) — Spearman vs W_REL=0.50, largest single rank move:
  W_REL=0.00: rho=0.965  max move=6 (CMC)
  W_REL=0.25: rho=0.992  max move=2 (CCI)
  W_REL=0.50: rho=1.000  max move=0 (ASP)
  W_REL=0.75: rho=0.986  max move=5 (WSP)
  W_REL=1.00: rho=0.957  max move=7 (CMC)
